# Computer Vision: Pełny Pipeline – od surowych pikseli do klasyfikacji i segmentacji

Ten notebook to obszerne, praktyczne uzupełnienie naszej prezentacji. Przełożymy tutaj całą teorię na kod w bibliotece PyTorch. Zobaczycie, jak definiować zaawansowane augmentacje z uwzględnieniem natury naszych danych, jak układać bloki warstw konwolucyjnych, jak kontrolować wymiary, aby uniknąć klątwy wymiarowości, oraz jak zbudować sieć do segmentacji.

---

## 0. Dlaczego w Deep Learningu i CV używamy GPU?

Zanim zaczniemy pisać architekturę, musimy ustalić, na czym będziemy to liczyć. W klasycznym programowaniu większość operacji wykonuje procesor (CPU). Dlaczego więc w Computer Vision tak bardzo polegamy na kartach graficznych (GPU)?

Odpowiedź leży w architekturze sprzętu:

* **CPU (Procesor):** Ma kilka lub kilkanaście bardzo szybkich i inteligentnych rdzeni. Jest świetny do wykonywania złożonych, sekwencyjnych instrukcji (jedna po drugiej).
* **GPU (Karta graficzna):** Posiada tysiące (często ponad 10 000) małych, prostszych rdzeni. Została zaprojektowana do renderowania grafiki, co sprowadza się do wykonywania tej samej, prostej operacji matematycznej na milionach pikseli w tym samym czasie.

**Co to ma do AI?**
Sieć neuronowa, a w szczególności warstwa konwolucyjna, pod spodem opiera się w niemal 100% na mnożeniu gigantycznych macierzy (czyli tysięcy pikseli pomnożonych przez wagi z filtrów). CPU robiłoby to piksel po pikselu, co trwałoby wieki. GPU bierze całą macierz zdjęcia i wykonuje tysiące mnożeń równolegle w ułamku sekundy. Bez GPU trening głębokich sieci wizyjnych byłby po prostu fizycznie niemożliwy w sensownym czasie.

## Uzupełnienie: jak rozumieć zadania w Computer Vision i dlaczego sieci są “przystosowane do obrazów”

Computer Vision (CV) zajmuje się tym, jak nauczyć model rozumieć dane obrazowe (piksele) i wyciągać z nich użyteczne wnioski. W praktyce CV to nie tylko jedna “klasa modelu”, ale cały zestaw typów zadań.

### Typy problemów w CV

**1) Klasyfikacja (Classification)**

- Wejście: obraz, wyjście: jedna etykieta (np. “kot”, “pies”, “litera A”).
- Duży plus: relatywnie proste etykietowanie.
- Duży minus: model dostaje informację “co to jest”, ale nie mówi automatycznie “gdzie”.

**2) Segmentacja (Segmentation)**

- Wejście: obraz, wyjście: mapa (na poziomie piksela) — każdy piksel dostaje klasę.
- Najczęściej mamy dwa “poziomy” bogactwa informacji:
  - *grupowanie* pikseli w plamy (czyli kształt/obszar),
  - *nadawanie etykiet* tym plamom (np. guz, tkanka zdrowa).
- Zastosowania: medycyna (segmentowanie tkanek/zmian), robotyka, autonomiczne systemy, rozpoznawanie pisma.
- Minusy: zwykle największe koszty adnotacji (wymagane maski per-piksel).

**3) Detekcja (Detection)**

- Wejście: obraz, wyjście: lokalizacje i klasy obiektów, najczęściej jako ramki (bounding boxes).
- Zwykle łatwiejsza niż segmentacja, bo etykietujemy “obszar” obiektu, a nie każdy piksel.
- Przykłady: znalezienie obiektu w scenie, wykrywanie anomalii na obrazach medycznych.

**4) Generatywne AI i XAI (Explainable AI)**

- *Generatywne AI* może tworzyć nowe treści (obrazy/napisy/odpowiedzi).
- *XAI* (wyjaśnialna sztuczna inteligencja) ma na celu tłumaczyć decyzje modelu: co dokładnie wpłynęło na klasyfikację/segmentację.

---

## Od MLP do CNN: co tak naprawdę “widzi” komputer

W klasycznych sieciach typu MLP (Multilayer Perceptron) wejściem jest wektor cech. Dla danych tablicowych to bywa naturalne, bo kolumny mają zwykle sensowną strukturę.

Dla obrazów sytuacja jest inna:

- obraz to **tensory** z wymiarami przestrzennymi, np. `H x W`,
- do tego dochodzi **głębokość kanałów** (najczęściej `C=3` dla RGB, albo `C=1` dla skanów w skali szarości).

Najczęstszy format w PyTorchu w praktyce to **NCHW**:

- `N` — batch size,
- `C` — kanały,
- `H` — wysokość,
- `W` — szerokość.

### Dlaczego Flatten jest problematyczne (i kiedy ma sens)

Jeśli weźmiesz obraz i zrobisz `Flatten`, to tracisz informację o sąsiedztwie pikseli.

Co to znaczy “w praktyce” dla modelu?

- Sieć w pełni połączona nie ma wbudowanego mechanizmu “lokalności” i “wagi współdzielonej” w różnych miejscach obrazu.
- Każda pozycja w wektorze staje się osobną cechą — przez to rośnie liczba parametrów i compute.
- Dodatkowo: model musi uczyć się zależności przestrzennych bez wrodzonej struktury (inductive bias) — to jest trudniejsze i mniej efektywne.

### Dlaczego konwolucja jest “naturalna” dla obrazów

Konwolucja działa przez:

- **lokalne okna (receptive field)** — kernel “patrzy” na niewielki fragment obrazu,
- **przesuwanie kernel’a po obrazie** — więc cechy są wykrywane w różnych lokalizacjach,
- **współdzielenie wag** między wszystkimi pozycjami — kernel wykrywa podobne wzorce wszędzie.

W efekcie:

- pierwsze warstwy uczą się prostych wzorców (krawędzie, tekstury),
- głębsze warstwy uczą się bardziej złożonych reprezentacji (kształty, części obiektów),
- wiele kanałów wyjściowych to wiele “map cech” (feature maps), z których każda koduje inny aspekt obrazu.

Atencja może rozwiązywać część problemów z długim kontekstem, ale konwolucja zwykle jest punktem startowym ze względu na efektywność i mocny bias przestrzenny.

---

## Pooling i Rule of Thumb dla CNN

Pooling zmniejsza rozdzielczość `H x W`, zwykle oszczędzając compute i zwiększając stabilność.

Typowe warianty:

- **max pooling** — wybiera lokalne maksimum (podkreśla “najsilniejsze” sygnały),
- **average pooling** — uśrednia (bardziej “wygładza” i może tłumić ostre artefakty).

Rule of Thumb, którą często się stosuje w projektowaniu CNN:

- gdy robisz pooling (redukujesz `H x W`), to zwykle zwiększasz liczbę kanałów (np. ~x2),
- dzięki temu utrzymujesz “pojemność reprezentacji”, ale kontrolujesz rozmiar obliczeń.

---

## Segmentacja i U-Net (dlaczego nie ma Flatten)

Segmentacja wymaga wyjścia w formie mapy o rozdzielczości przestrzennej.

Dlatego U-Net (i ogólnie sieci w pełni konwolucyjne) robią:

- encoder: konwolucje + downsampling,
- decoder: upsampling back do wysokiej rozdzielczości,
- skip connections: przerzucenie szczegółów z wczesnych warstw do dekodera.

Skip connections pomagają modelowi jednocześnie:

- “wiedzieć co” (z głębi sieci),
- “wiedzieć gdzie” (z zachowanych detali z lewej strony).

---

## Ograniczenia konwolucji i rola atencji

Konwolucja ma skończone pole widzenia zależne od głębokości i parametrów warstw. Gdy potrzebujesz relacji na bardzo dużym dystansie w obrazie, konwolucja może być mniej wygodna.

Wtedy wchodzi atencja (globalna albo hybrydowa), bo daje mechanizm zależności “dalekiego zasięgu”.

W praktyce dziś często buduje się mieszaniny: CNN dla lokalnych wzorców + atencja dla kontekstu.


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms

# Ustawienie urządzenia (Device configuration)
# PyTorch pozwala nam łatwo przerzucać dane między RAMem procesora a VRAMem karty graficznej.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Będziemy trenować nasz model używając: {device.type.upper()}")


Będziemy trenować nasz model używając: CUDA


## 1. Pre-processing, Data Augmentation i Noise Injection

Augmentacja to nasza pierwsza linia obrony przed przeuczeniem (overfittingiem). Różne problemy wymagają zupełnie różnego podejścia do transformacji. W PyTorchu używamy `transforms.Compose`, aby złożyć operacje w jeden pipeline, który jest aplikowany w locie dla każdego wczytywanego zdjęcia.

**Przegląd wykorzystywanych transformacji:**

1. **`RandomResizedCrop(size)`**: Losowo wycina fragment obrazka i skaluje go do podanego rozmiaru. Zmusza model do podejmowania decyzji na podstawie detali (np. połowy twarzy czy ucha), a nie zapamiętywania ogólnego zarysu zdjęcia.
2. **`RandomHorizontalFlip(p)`**: Z zadanym prawdopodobieństwem odbija obraz w poziomie. Idealne dla obiektów w przestrzeni (np. zwierzęta, samochody). Złe dla analizy ruchu kierunkowego.
3. **`RandomRotation(degrees)`**: Obraca zdjęcie o losowy kąt. Bardzo przydatne przy skanach dokumentów, rozpoznawaniu pisma czy gestach dłoni, gdzie ułożenie rzadko jest idealnie proste.
4. **`ColorJitter(brightness, contrast, saturation)`**: Zmienia właściwości kolorystyczne pikseli. Świetnie symuluje różne pory dnia i różną jakość aparatów.
   **Uwaga:** Unikamy tego przy ustandaryzowanych danych medycznych (np. rentgen), gdzie kontrast ma kluczowe znaczenie diagnostyczne.
5. **`AddGaussianNoise` (Niestandardowa klasa)**: Celowe wprowadzanie szumu, tzw. noise injection. Pozwala uodpornić model na zdjęcia gorszej jakości i zakłócenia z matryc tanich aparatów.
6. **`ToTensor()`**: Zmienia obraz z formatu PIL (wartości 0-255) na Tensor matematyczny (wartości 0.0 - 1.0) i zmienia układ z [Wysokość, Szerokość, Kanały] na format optymalny dla PyTorch: **[Kanały, Wysokość, Szerokość]**.
7. **`Normalize(mean, std)`**: Zmiana rozkładu danych tak, aby średnia wynosiła 0. Pomaga algorytmom optymalizacyjnym (jak Adam czy SGD) znacznie szybciej zbiegać do najlepszego wyniku.

Preprocessing/augmentacje to operacje wykonywane *przed* treningiem (na etapie wczytywania danych). Ich celem jest zmniejszenie przeuczenia i zasymulowanie tego, jak dane będą wyglądały w świecie rzeczywistym.

### Najważniejsze typy augmentacji w CV

**Flip (odbicia)**

- `RandomHorizontalFlip` / `RandomVerticalFlip` zwiększają wariancję ułożenia obiektu.
- Mają sens, gdy znak odwrócenia nie zmienia sensu etykiety (np. zwierzęta, wiele obiektów w scenach).
- Mogą szkodzić w zadaniach, gdzie kierunek ma znaczenie (np. pewne typy ruchu lub orientacje w danych).

**Rotation (rotacje)**

- `RandomRotation` pomaga, gdy obiekty mogą występować pod różnymi kątami (np. pismo odręczne, dłonie, dokumenty skanowane pod różnym nachyleniem).

**Crop (wycinanie fragmentów)**

- `RandomResizedCrop`/`RandomCrop` zmusza model do uczenia się na podstawie części obrazu — przez to rośnie odporność na fakt, że obiekt może być tylko częściowo widoczny.

**Color Jitter (zmiany koloru/kontrastu)**

- `ColorJitter` symuluje zmienność oświetlenia, jakości aparatów i pory dnia.
- Uwaga: w danych, gdzie kontrast/kolory mają znaczenie diagnostyczne lub są silnie standaryzowane (medycyna), zwykle jest to ryzykowne.

**Noise injection (celowe zaszumianie)**

- `AddGaussianNoise` (albo inny typ szumu) uczy model odporności na degradację jakości obrazu (tańsze aparaty, kompresja, zakłócenia).

### Rekomendowana zasada praktyczna

- Augmentacje robimy tylko na zbiorze treningowym.
- Na walidacji/testach używamy “czystych” przekształceń (np. resize + normalize), żeby metryka była uczciwa.

W tym notebooku masz już działający przykład pipeline’u augmentacyjnego (komórka z `train_transform` i `test_transform`). Powyższa sekcja ma Ci pomóc podejmować decyzje, *dlaczego* i *kiedy* konkretna augmentacja ma sens.



In [ ]:
# Customowa transformacja do wstrzykiwania szumu
class AddGaussianNoise(object):
    def __init__(self, mean=0.0, std=1.0):
        self.std = std
        self.mean = mean

    def __call__(self, tensor):
        # Generujemy tensor szumu o takich samych wymiarach jak wejście
        noise = torch.randn(tensor.size()) * self.std + self.mean
        # Dodajemy szum i używamy clamp, aby wartości pikseli pozostały w granicach 0-1
        return torch.clamp(tensor + noise, 0.0, 1.0)


# Pipeline treningowy (Z pełną, agresywną augmentacją)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    AddGaussianNoise(mean=0.0, std=0.05),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Pipeline walidacyjny/testowy
# UWAGA: Na zbiorze testowym NIE ROBIMY augmentacji. Chcemy sprawdzić model na "czystych" i rzeczywistych danych,
# więc tylko zmieniamy rozmiar i ujednolicamy format.
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Pipelines transformacji wczytane pomyślnie.")

# Przykładowy (dokumentacyjny) wariant augmentacji:
# - pokazuje typowe “klocki” z wykładu,
# - dodaje też flip pionowy (dla kompletności względem opisu).
#
# UWAGA: to jest przykład konfiguracji — dopasuj go do Twojego zadania.

def build_train_transform(img_size=(224, 224), noise_std=0.05, do_color_jitter=True):
    ops = [
        transforms.RandomResizedCrop(size=img_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(degrees=15),
    ]

    if do_color_jitter:
        ops.append(transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1))

    ops.extend([
        transforms.ToTensor(),
        AddGaussianNoise(mean=0.0, std=noise_std),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    return transforms.Compose(ops)


train_transform_v2 = build_train_transform(img_size=(224, 224), noise_std=0.05, do_color_jitter=True)

# Walidacja/test: bez augmentacji, tylko standaryzacja
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("train_transform_v2 oraz val_test_transform zbudowane.")



Pipelines transformacji wczytane pomyślnie.
train_transform_v2 oraz val_test_transform zbudowane.


## 2. Anatomia Warstw w Computer Vision (Krok po Kroku)

Zanim zbudujemy pełną sieć, prześwietlmy najważniejsze "klocki", z których jest ona budowana. Musimy wiedzieć, jak każdy z nich zmienia kształt naszego Tensora.

Zawsze operujemy na standardowym formacie wymiarów: **[Rozmiar Batcha, Liczba Kanałów, Wysokość, Szerokość]** (często oznaczane jako N, C, H, W).


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms

# Symulujemy JEDNO zdjęcie RGB 64x64 w batchu
# Kształt: [Batch=1, Channels=3, Height=64, Width=64]
x = torch.randn(1, 3, 64, 64)
print(f"INPUT (Oryginalne zdjęcie): {x.shape}\n")

# --- 1. nn.Conv2d: Ekstrakcja cech ---
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
out_conv = conv(x)
print(f"Po nn.Conv2d:               {out_conv.shape}")
print("   -> Kanały w górę (3 -> 16), przestrzeń (H, W) bez zmian.\n")

# --- 2. nn.MaxPool2d: Redukcja rozdzielczości (Downsampling) ---
pool = nn.MaxPool2d(kernel_size=2, stride=2)
out_pool = pool(out_conv)
print(f"Po nn.MaxPool2d:            {out_pool.shape}")
print("   -> Kanały bez zmian, wysokość/szerokość ucięta o połowę (64 -> 32).\n")

# --- 3. nn.BatchNorm2d: Stabilizacja Treningu ---
batch_norm = nn.BatchNorm2d(num_features=16)
out_bn = batch_norm(out_pool)
print(f"Po nn.BatchNorm2d:          {out_bn.shape}")
print("   -> Wymiary absolutnie bez zmian. Zmieniają się tylko matematyczne wartości w środku Tensora.\n")

# --- 4. nn.Flatten: Płaskowanie struktur ---
flatten = nn.Flatten()
out_flatten = flatten(out_bn)
print(f"Po nn.Flatten:              {out_flatten.shape}")
print("   -> Wynik to wymnożenie: 16 kanałów * 32 wys. * 32 szer. = 16384 cech dla jednego zdjęcia.\n")

# --- 5. nn.Linear (Fully Connected): Decyzyjność ---
linear = nn.Linear(in_features=16384, out_features=10)
out_linear = linear(out_flatten)
print(f"Po nn.Linear:               {out_linear.shape}")
print("   -> Finalnie otrzymujemy 10 wartości dla naszego 1 zdjęcia (np. prawdopodobieństwa dla każdej z klas).")

# --- Dodatkowe uzupełnienie: liczby dla Flatten + mała demonstracja konwolucji ---

# Flatten: szybki rachunek rozmiarów wejścia
# (żeby zrozumieć, czemu “klątwa wymiarowości” realnie boli w praktyce)

def num_elements(h, w, c):
    return h * w * c

# MNIST (28x28, skala szarości)
mnist_elems = num_elements(28, 28, 1)
print("MNIST 28x28x1 ->", mnist_elems, "liczb")

# Typowy obrazek RGB 224x224
img_elems = num_elements(224, 224, 3)
print("224x224x3 ->", img_elems, "liczb")

# “Prawie 4K”: np. 3840x2160
h4k, w4k, c = 2160, 3840, 3
img4k_elems = num_elements(h4k, w4k, c)
print(f"4K ({h4k}x{w4k}x{c}) ->", img4k_elems, "liczb")

# Szacunek pamięci na same wejście jako float32 (4 bajty/liczbę)
bytes_est = img4k_elems * 4
print(f"Float32 pamięć dla 4K ~ {bytes_est / (1024**2):.1f} MB (tylko dla wejścia)")


import torch.nn.functional as F

# Konwolucja: prosta intuicja na ręcznym kernel’u + porównanie map aktywacji

# Symulujemy mały obrazek (skala szarości)
# Shape: [N=1, C=1, H=5, W=5]
img = torch.tensor([
    [
        [0., 0., 0., 0., 0.],
        [0., 1., 1., 1., 0.],
        [0., 1., 2., 1., 0.],
        [0., 1., 1., 1., 0.],
        [0., 0., 0., 0., 0.],
    ]
]).unsqueeze(0) # Dodanie wymiaru dla batcha

# Kernel detekcji krawędzi (prosty “sobel-like” / laplace-ish)
# Shape kernel’u w conv2d: [out_channels, in_channels, kH, kW]
edge_kernel = torch.tensor([
    [
        [-1., -1., -1.],
        [-1., 8., -1.],
        [-1., -1., -1.],
    ]
]).unsqueeze(0) # Dodanie wymiaru dla in_channels (powinno być [1, 1, 3, 3])

# padding=1, stride=1 => H i W pozostają takie same
out_edge = F.conv2d(img, edge_kernel, padding=1)
print("Wejście shape:", img.shape)
print("Wyjście (1 kanał) shape:", out_edge.shape)
print("Wyjście (wartości):\n", out_edge[0, 0])

# Wersja “feature extraction” z wieloma kanałami wyjściowymi
conv_multi = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1, bias=False)
feat_maps = conv_multi(img)  # [1, 4, 5, 5]
print("Feature maps shape (C_out=4):", feat_maps.shape)

# Pooling: max vs average
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

p_max = max_pool(out_edge)
p_avg = avg_pool(out_edge)
print("MaxPool shape:", p_max.shape)
print("AvgPool shape:", p_avg.shape)
print("MaxPool (wartości):\n", p_max[0, 0])
print("AvgPool (wartości):\n", p_avg[0, 0])

INPUT (Oryginalne zdjęcie): torch.Size([1, 3, 64, 64])

Po nn.Conv2d:               torch.Size([1, 16, 64, 64])
   -> Kanały w górę (3 -> 16), przestrzeń (H, W) bez zmian.

Po nn.MaxPool2d:            torch.Size([1, 16, 32, 32])
   -> Kanały bez zmian, wysokość/szerokość ucięta o połowę (64 -> 32).

Po nn.BatchNorm2d:          torch.Size([1, 16, 32, 32])
   -> Wymiary absolutnie bez zmian. Zmieniają się tylko matematyczne wartości w środku Tensora.

Po nn.Flatten:              torch.Size([1, 16384])
   -> Wynik to wymnożenie: 16 kanałów * 32 wys. * 32 szer. = 16384 cech dla jednego zdjęcia.

Po nn.Linear:               torch.Size([1, 10])
   -> Finalnie otrzymujemy 10 wartości dla naszego 1 zdjęcia (np. prawdopodobieństwa dla każdej z klas).
MNIST 28x28x1 -> 784 liczb
224x224x3 -> 150528 liczb
4K (2160x3840x3) -> 24883200 liczb
Float32 pamięć dla 4K ~ 94.9 MB (tylko dla wejścia)
Wejście shape: torch.Size([1, 1, 5, 5])
Wyjście (1 kanał) shape: torch.Size([1, 1, 5, 5])
Wyjście (wartości)

## 3. Pełna Architektura Klasyfikacyjna (CNN)

Mając tę wiedzę, złóżmy pełny model. Podstawowa zasada architektoniczna, o której wspominaliśmy podczas prezentacji, wygląda tak:
**Z każdym zastosowaniem poolingu (redukcją wymiarowości przestrzennej), zwiększamy liczbę filtrów (kanałów) przeważnie razy dwa.**

Ten proces redukuje obraz przestrzenie, zmuszając sieć do wyciągania coraz bardziej abstrakcyjnych, gęstych informacji, aż będą gotowe do łatwego "spłaszczenia".


In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # --- SEKCJA 1: EKSTRAKCJA CECH (Feature Extractor) ---
        # Analizujemy sąsiedztwo pikseli w 2D, budując mapy ważnych cech
        self.features = nn.Sequential(
            # Blok 1
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Blok 2: Redukujemy przestrzeń, więc podwajamy kanały (64 -> 128)
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Blok 3: Kolejna redukcja, kolejne podwojenie kanałów (128 -> 256)
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # --- SEKCJA 2: KLASYFIKATOR (MLP) ---
        # Wymiary przed Flattenem dla obrazu 224x224:
        # 3 MaxPoole powodują: 224 -> 112 -> 56 -> 28
        # Kanały: 256
        # Zatem: 256 * 28 * 28 = 200704
        self.classifier = nn.Sequential(
            nn.Flatten(),

            # Dropout gaszący 50% neuronów. Zmusza model do redundancji w uczeniu i zapobiega overfittingowi.
            nn.Dropout(p=0.5),
            nn.Linear(in_features=256 * 28 * 28, out_features=512),
            nn.ReLU(),

            nn.Dropout(p=0.5),
            nn.Linear(in_features=512, out_features=num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Testowanie przepływu całego batcha przez naszą sieć
model_cnn = CNNClassifier(num_classes=10)
dummy_batch = torch.randn(8, 3, 224, 224)  # Batch 8 zdjęć RGB w rozdzielczości 224x224

out_cnn = model_cnn(dummy_batch)
print(f"Kształt przed wejściem do pełnego CNN: {dummy_batch.shape}")
print(f"Wyjście po całej analizie i spłaszczeniu: {out_cnn.shape} -> 8 wektorów decyzyjnych po 10 klas.")


Kształt przed wejściem do pełnego CNN: torch.Size([8, 3, 224, 224])
Wyjście po całej analizie i spłaszczeniu: torch.Size([8, 10]) -> 8 wektorów decyzyjnych po 10 klas.


## 4. Architektura Segmentacyjna (U-Net) – Całkowity brak Flattena

Klasyfikacja ma jeden cel: połączyć setki tysięcy pikseli w jedną ogólną etykietę. Ale co, jeśli rozwiązujemy problem medyczny i musimy wysegmentować guza na rezonansie? Tu interesuje nas idealna mapa przestrzenna: chcemy na wyjściu otrzymać maskę o dokładnie takich samych rozmiarach co zdjęcie wejściowe.

**Zasady dla sieci segmentacyjnych (Fully Convolutional Networks):**

1. **Omijamy warstwy Fully Connected i Flatten całkowicie!** Spłaszczenie wymiarów (1D) sprawia, że sieć bezpowrotnie traci informację przestrzenną.
2. Zamiast zatrzymywać się na wąskiej reprezentacji cech, wprowadzamy etap **Dekodera**.
3. Wykorzystujemy `nn.ConvTranspose2d` (czasami nazywaną dekonwolucją), która fizycznie powiększa (rozdyma) rozdzielczość obrazu z powrotem do rozmiaru początkowego.
4. Wyjście to konwolucja z kernelem wielkości 1x1, która spłaszcza głębokość kanałów do liczby klas, nie psując ułożenia przestrzennego 2D.


In [ ]:
class SimpleSegmentationUNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        # --- ENKODER ---
        # Zakładamy skan medyczny w skali szarości (1 kanał in)
        self.enc1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # --- BOTTLENECK ---
        self.bottleneck = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)

        # --- DEKODER ---
        # Transpose Convolution fizycznie rozszerza wymiary przestrzenne x2
        self.upconv = nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=2, stride=2)

        # Wygładzenie cech po powiększeniu
        self.dec1 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)

        # --- WARSTWA WYJŚCIOWA ---
        # Konwolucja 1x1 mapuje 64 abstrakcyjne kanały na 3 klasy (per-piksel)
        self.final_conv = nn.Conv2d(in_channels=64, out_channels=num_classes, kernel_size=1)

    def forward(self, x):
        # 1. Enkoder
        e1 = self.enc1(x)      # Mapy zostają w rozdzielczości wejścia
        p1 = self.pool(e1)     # Rozdzielczość ucięta o połowę

        # 2. Środek
        b = self.bottleneck(p1)

        # 3. Dekoder
        up = self.upconv(b)   # Rozdzielczość przestrzenna odzyskana (x2)

        # [W pełnym U-NeCie: tutaj byłaby Skip Connection]

        d1 = self.dec1(up)

        # 4. Decyzja per piksel (brak Flattena)
        out = self.final_conv(d1)
        return out


# Przetestujmy model segmentacyjny z symulowanym skanem mózgu MRI
seg_model = SimpleSegmentationUNet(num_classes=3)  # 3 klasy wyjściowe
medical_scan = torch.randn(2, 1, 128, 128)          # Batch: 2 skany, 128x128

mask_output = seg_model(medical_scan)

print(f"Wejście (Skan MRI):            {medical_scan.shape}")
print(f"Wyjście (Wysegmentowana Mapa): {mask_output.shape}")
print("\nSukces! Otrzymaliśmy dokładnie tę samą rozdzielczość (128x128).")
print("Ilość kanałów na wyjściu (3) to logits/prawdopodobieństwa przynależności pojedynczego piksela do każdej z klas.")

# --- Dodatkowe uzupełnienie: U-Net z jednym skip connection (dla intuicji) ---

# U-Net z skip connections (mini wersja)
# To kodowa reprezentacja “przerzuconych detalów”: wiemy, co jest (z głębi), i gdzie (z wczesnych warstw).

class MiniUNetSkip(nn.Module):
    def __init__(self, num_classes=3, in_channels=1, base=16):
        super().__init__()

        # Encoder
        self.enc1 = nn.Conv2d(in_channels, base, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = nn.Conv2d(base, base * 2, kernel_size=3, padding=1)

        # Decoder
        self.upconv = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        # Po concat mamy 2*base kanałów
        self.dec1 = nn.Conv2d(base * 2, base, kernel_size=3, padding=1)

        # Wyjście per-piksel
        self.final_conv = nn.Conv2d(base, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)            # [N, base, H, W]
        p1 = self.pool(e1)          # [N, base, H/2, W/2]
        b = self.bottleneck(p1)    # [N, base*2, H/2, W/2]
        up = self.upconv(b)        # [N, base, H, W]

        # Skip connection: concat po kanałach
        cat = torch.cat([up, e1], dim=1)  # [N, base*2, H, W]
        d1 = self.dec1(cat)              # [N, base, H, W]
        out = self.final_conv(d1)       # [N, num_classes, H, W]
        return out


# Test kształtów
mini_unet = MiniUNetSkip(num_classes=3, in_channels=1, base=16)
scan = torch.randn(2, 1, 128, 128)
mask = mini_unet(scan)

print("Wejście:", scan.shape)
print("Wyjście:", mask.shape)



Wejście (Skan MRI):            torch.Size([2, 1, 128, 128])
Wyjście (Wysegmentowana Mapa): torch.Size([2, 3, 128, 128])

Sukces! Otrzymaliśmy dokładnie tę samą rozdzielczość (128x128).
Ilość kanałów na wyjściu (3) to logits/prawdopodobieństwa przynależności pojedynczego piksela do każdej z klas.
Wejście: torch.Size([2, 1, 128, 128])
Wyjście: torch.Size([2, 3, 128, 128])
